# Notebook 01 - Justifikasi Rasio Split Data 80:20

**Revisi penguji poin 3: "Kenapa splitting data pakai 80:20?"**

---

## 1. Pernyataan masalah

Pada notebook V2, pembagian data dilakukan dengan satu baris:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
```

Angka `test_size=0.20` dipakai **tanpa dasar eksperimen apa pun** - hanya mengikuti
kebiasaan umum. Ini adalah kelemahan metodologis yang sah dipersoalkan penguji:
sebuah keputusan desain penelitian tidak boleh berstatus "kebiasaan", ia harus
berstatus **hasil pengujian**.

Notebook ini menutup celah tersebut dengan menyediakan **bukti empiris + analisis
statistik** yang menunjukkan bahwa 80:20 memang rasio yang rasional untuk dataset
dan tujuan penelitian ini (deteksi dini diabetes, metrik prioritas = **recall**).

## 2. Kerangka argumen: trade-off bias-variance pada pembagian data

Rasio split mengendalikan dua hal yang saling bertentangan:

| Arah | Efek pada model | Efek pada estimasi kinerja |
|---|---|---|
| Data latih diperbesar (test kecil) | Model **lebih baik** (bias turun, model melihat lebih banyak pola) | Estimasi kinerja **kurang presisi** (test kecil -> variansi tinggi, confidence interval lebar) |
| Data uji diperbesar (train kecil) | Model **lebih lemah** (bias naik, kekurangan data belajar) | Estimasi kinerja **lebih presisi** (margin of error mengecil) |

Jadi memilih rasio split = memilih titik kompromi antara **kualitas model** dan
**kepercayaan pada angka hasil ujinya**. Rasio yang benar adalah rasio yang berada
tepat setelah kurva performa mendatar (*plateau*) tetapi sebelum margin of error
estimasi membengkak.

## 3. Empat eksperimen dalam notebook ini

| # | Eksperimen | Pertanyaan yang dijawab |
|---|---|---|
| 1 | **Sweep rasio split** 50:50 s.d. 90:10 x 5 seed x 3 model | Apakah menambah data latih di atas 80% masih menaikkan performa? |
| 2 | **Analisis Margin of Error (MoE)** 95% pada recall | Seberapa presisi angka recall yang dilaporkan pada tiap ukuran data uji? |
| 3 | **Learning curve** (scoring recall, 5-fold CV) | Pada berapa banyak data latih kurva belajar mencapai plateau? |
| 4 | **Holdout 80:20 vs 5-Fold vs 10-Fold vs Repeated CV** | Apakah estimasi holdout 80:20 bias dibanding cross-validation? |

Seluruh notebook mengikuti kontrak `_SPEC_BERSAMA.md` (preamble, pabrik pipeline,
fungsi evaluasi, dan penamaan file output identik lintas notebook).

In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---

## 4. Konfigurasi eksperimen

Sesuai `_SPEC_BERSAMA.md` bagian 3 (strategi biaya komputasi), notebook menyediakan
saklar `MODE_CEPAT`:

- `MODE_CEPAT = True` -> memakai **subsample stratified 30.000 baris**. Dipakai untuk
  memastikan seluruh notebook jalan dari atas ke bawah dalam hitungan menit.
- `MODE_CEPAT = False` -> memakai **dataset penuh 96.146 baris**. **Wajib dipakai untuk
  angka final yang ditulis di skripsi.**

Rasio yang diuji (proporsi **data uji**): 50%, 40%, 30%, 25%, 20%, 15%, 10% - yaitu
split 50:50, 60:40, 70:30, 75:25, **80:20**, 85:15, dan 90:10. Setiap rasio diulang
pada **5 seed** (42-46) agar variasi akibat keberuntungan pembagian data dapat diukur
sebagai standar deviasi, bukan disembunyikan.

In [ ]:
# ============================================================
# CELL 7: Konstanta Eksperimen Rasio Split + Estimasi Waktu
# ============================================================
MODE_CEPAT  = True      # True = subsample cepat (uji coba) | False = data penuh (angka final skripsi)
N_SUBSAMPLE = 30000

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    if n >= len(X): return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

# Proporsi DATA UJI yang diuji (0.20 = split 80:20)
DAFTAR_RASIO = [0.50, 0.40, 0.30, 0.25, 0.20, 0.15, 0.10]
N_SEED       = 5
DAFTAR_SEED  = [RANDOM_STATE + i for i in range(N_SEED)]   # 42, 43, 44, 45, 46
DAFTAR_MODEL = ['Random Forest', 'KNN', 'SVM (Linear)']
RASIO_SKRIPSI = 0.20                                        # rasio yang dipakai V2 dan diuji di sini

def label_rasio(r):
    return f'{int(round((1 - r) * 100))}:{int(round(r * 100))}'

if MODE_CEPAT:
    X_eks, y_eks = ambil_subsample(X_all, y_all, N_SUBSAMPLE)
    LABEL_MODE, DETIK_PER_FIT = 'MODE_CEPAT (subsample stratified)', 1.5
else:
    X_eks, y_eks = X_all.copy(), y_all.copy()
    LABEL_MODE, DETIK_PER_FIT = 'MODE PENUH (dataset lengkap)', 5.0

N_FIT_EKS1 = len(DAFTAR_RASIO) * N_SEED * len(DAFTAR_MODEL)
N_FIT_EKS3 = len(DAFTAR_MODEL) * 10 * 5          # learning curve: 10 titik x 5 fold
N_FIT_EKS4 = 5 + 10 + 25                          # 5-fold + 10-fold + repeated 5x5
N_FIT_TOTAL = N_FIT_EKS1 + N_FIT_EKS3 + N_FIT_EKS4

garis('KONFIGURASI EKSPERIMEN RASIO SPLIT')
print(f'Mode                 : {LABEL_MODE}')
print(f'Ukuran data dipakai  : {len(X_eks):,} baris '
      f'({int(y_eks.sum()):,} positif / {y_eks.mean()*100:.2f}%)')
print(f'Dataset penuh        : {len(X_all):,} baris ({int(y_all.sum()):,} positif)')
print(f'Rasio uji diuji      : {[label_rasio(r) for r in DAFTAR_RASIO]}')
print(f'Seed                 : {DAFTAR_SEED}')
print(f'Model                : {DAFTAR_MODEL}')
print('-' * 70)
print(f'Perkiraan jumlah fit : Eks-1 {N_FIT_EKS1} | Eks-3 {N_FIT_EKS3} | '
      f'Eks-4 {N_FIT_EKS4} | TOTAL {N_FIT_TOTAL}')
print(f'Estimasi waktu total : ~{N_FIT_TOTAL * DETIK_PER_FIT / 60:.1f} menit '
      f'(asumsi {DETIK_PER_FIT:.1f} detik/fit pada Colab CPU)')
if MODE_CEPAT:
    print('CATATAN: hasil di bawah adalah pratinjau. Untuk angka final skripsi,')
    print('         set MODE_CEPAT = False lalu jalankan ulang notebook ini.')

---

# EKSPERIMEN 1 - Sweep Rasio Split (7 rasio x 5 seed x 3 model)

**Pertanyaan:** apakah menambah porsi data latih terus-menerus menaikkan performa,
atau ada titik jenuh?

**Desain:** untuk setiap rasio dan setiap seed dilakukan `train_test_split` **stratified**
(proporsi kelas dipertahankan), lalu ketiga pipeline (Scaler -> SMOTE -> Classifier,
hyperparameter V2) dilatih pada data latih dan dievaluasi pada data uji dengan
`evaluasi_holdout`. Metrik utama = **recall pada threshold Youden**, sesuai konteks
skrining medis (biaya *false negative* jauh lebih besar daripada *false positive*).

Total: 7 x 5 x 3 = **105 kali pelatihan-evaluasi**.

In [ ]:
# ============================================================
# CELL 8: Eksperimen 1 - Sweep rasio split x seed x model
# ============================================================
garis('EKSPERIMEN 1: SWEEP RASIO SPLIT')
print(f'Total kombinasi: {N_FIT_EKS1} (progres dicetak per kombinasi)')
print('-' * 70)

baris_hasil = []
t_mulai_eks1 = time.time()

for i_r, rasio in enumerate(DAFTAR_RASIO, 1):
    print(f'[RASIO {i_r}/{len(DAFTAR_RASIO)}] split {label_rasio(rasio)} '
          f'(test_size={rasio:.2f})')
    for seed in DAFTAR_SEED:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_eks, y_eks, test_size=rasio, stratify=y_eks, random_state=seed)
        for nama_model in DAFTAR_MODEL:
            t0 = time.time()
            model = PABRIK_MODEL[nama_model](pakai_smote=True)
            hasil = evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True)
            baris_hasil.append({
                'model'          : nama_model,
                'rasio_uji'      : rasio,
                'rasio_label'    : label_rasio(rasio),
                'proporsi_latih' : round(1 - rasio, 2),
                'seed'           : seed,
                'n_train'        : int(len(X_tr)),
                'n_test'         : int(len(X_te)),
                'n_positif_test' : int(y_te.sum()),
                **hasil,
            })
            print(f'   seed={seed} {nama_model:<14} '
                  f'recall={hasil["recall_tuned"]:.4f} '
                  f'f1={hasil["f1_tuned"]:.4f} '
                  f'auc={hasil["roc_auc_tuned"]:.4f} '
                  f'({time.time() - t0:.1f}s)')

print('-' * 70)
print(f'Eksperimen 1 selesai dalam {(time.time() - t_mulai_eks1)/60:.1f} menit '
      f'({len(baris_hasil)} baris hasil).')

df_split_detail = pd.DataFrame(baris_hasil)
simpan_tabel(df_split_detail, 'tabel_rasio_split_detail', tampilkan=False)
display(df_split_detail.head(10))

In [ ]:
# ============================================================
# CELL 9: Agregasi hasil sweep (mean +/- std antar seed)
# ============================================================
METRIK_EKS1 = ['recall_tuned', 'f1_tuned', 'roc_auc_tuned',
               'precision_tuned', 'accuracy_tuned']

baris_ringkas = []
for (nama_model, rasio), g in df_split_detail.groupby(['model', 'rasio_uji']):
    d = {
        'model'          : nama_model,
        'rasio_uji'      : rasio,
        'rasio_label'    : label_rasio(rasio),
        'proporsi_latih' : round(1 - rasio, 2),
        'n_train'        : int(g['n_train'].mean()),
        'n_test'         : int(g['n_test'].mean()),
        'n_positif_test' : int(g['n_positif_test'].mean()),
    }
    for m in METRIK_EKS1:
        d[f'{m}_mean'] = float(g[m].mean())
        d[f'{m}_std']  = float(g[m].std(ddof=1))
    d['waktu_latih_s_mean'] = float(g['waktu_latih_s'].mean())
    baris_ringkas.append(d)

df_split_ringkas = (pd.DataFrame(baris_ringkas)
                    .sort_values(['model', 'proporsi_latih'])
                    .reset_index(drop=True))
simpan_tabel(df_split_ringkas.round(4), 'tabel_rasio_split_ringkas', tampilkan=False)

garis('RINGKASAN: RECALL (mean +/- std dari 5 seed) PER RASIO')
print(f'{"Model":<15}{"Split":<9}{"n_train":>9}{"n_test":>9}'
      f'{"recall":>18}{"F1":>18}{"ROC-AUC":>18}')
print('-' * 96)
for nama_model in DAFTAR_MODEL:
    sub = df_split_ringkas[df_split_ringkas['model'] == nama_model]
    for _, r in sub.iterrows():
        print(f'{r["model"]:<15}{r["rasio_label"]:<9}{r["n_train"]:>9,}{r["n_test"]:>9,}'
              f'{r["recall_tuned_mean"]:>11.4f} +/-{r["recall_tuned_std"]:.4f}'
              f'{r["f1_tuned_mean"]:>11.4f} +/-{r["f1_tuned_std"]:.4f}'
              f'{r["roc_auc_tuned_mean"]:>11.4f} +/-{r["roc_auc_tuned_std"]:.4f}')
    print('-' * 96)

garis('SELISIH PERFORMA ANTAR RASIO (indikasi plateau)')
for nama_model in DAFTAR_MODEL:
    sub = df_split_ringkas[df_split_ringkas['model'] == nama_model].set_index('rasio_uji')
    r50, r20, r10 = sub.loc[0.50], sub.loc[0.20], sub.loc[0.10]
    print(f'{nama_model}:')
    print(f'   recall 50:50 -> 80:20 : {r50["recall_tuned_mean"]:.4f} -> '
          f'{r20["recall_tuned_mean"]:.4f} (delta {(r20["recall_tuned_mean"]-r50["recall_tuned_mean"])*100:+.2f} poin)')
    print(f'   recall 80:20 -> 90:10 : {r20["recall_tuned_mean"]:.4f} -> '
          f'{r10["recall_tuned_mean"]:.4f} (delta {(r10["recall_tuned_mean"]-r20["recall_tuned_mean"])*100:+.2f} poin)')
    print(f'   std antar seed 80:20  : {r20["recall_tuned_std"]:.4f} | '
          f'90:10 : {r10["recall_tuned_std"]:.4f} '
          f'(naik {((r10["recall_tuned_std"]+1e-12)/(r20["recall_tuned_std"]+1e-12)):.2f}x)')

## Visualisasi 1 - Metrik vs proporsi data latih

Sumbu-X adalah **proporsi data latih** (50% s.d. 90%). *Error bar* menampilkan standar
deviasi antar 5 seed. Garis putus-putus oranye menandai posisi **80% data latih (80:20)**.

Yang perlu dilihat: setelah 70-80% data latih, kurva mendatar - penambahan data latih
tidak lagi diikuti kenaikan metrik yang berarti (kenaikan lebih kecil daripada lebar
error bar-nya sendiri, artinya tidak signifikan).

In [ ]:
# ============================================================
# CELL 10: Visualisasi 1 - Metrik vs proporsi data latih
# ============================================================
METRIK_PLOT = [
    ('recall_tuned',    'Recall (Sensitivitas)'),
    ('f1_tuned',        'F1-Score'),
    ('roc_auc_tuned',   'ROC-AUC'),
    ('precision_tuned', 'Precision'),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, (metrik, judul) in zip(axes.ravel(), METRIK_PLOT):
    for nama_model in DAFTAR_MODEL:
        sub = (df_split_ringkas[df_split_ringkas['model'] == nama_model]
               .sort_values('proporsi_latih'))
        ax.errorbar(sub['proporsi_latih'] * 100,
                    sub[f'{metrik}_mean'],
                    yerr=sub[f'{metrik}_std'],
                    marker='o', markersize=7, linewidth=2.2, capsize=5,
                    color=WARNA_MODEL[nama_model], label=nama_model)
    ax.axvline((1 - RASIO_SKRIPSI) * 100, color=WARNA_AKSEN,
               linestyle='--', linewidth=2.2, alpha=0.9)
    ax.text((1 - RASIO_SKRIPSI) * 100 + 0.6, ax.get_ylim()[0], ' 80:20',
            color=WARNA_AKSEN, fontsize=10, fontweight='bold', va='bottom')
    ax.set_title(judul, fontweight='bold')
    ax.set_xlabel('Proporsi data latih (%)')
    ax.set_ylabel(judul)
    ax.set_xticks([50, 60, 70, 75, 80, 85, 90])
    ax.legend(fontsize=9, loc='best')

plt.suptitle('Eksperimen 1: Performa Model vs Proporsi Data Latih '
             '(rata-rata 5 seed, error bar = std antar seed)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
simpan_gambar('split_metrik_vs_rasio')
plt.show()

garis('BACAAN GRAFIK')
for nama_model in DAFTAR_MODEL:
    sub = (df_split_ringkas[df_split_ringkas['model'] == nama_model]
           .sort_values('proporsi_latih'))
    idx_max = sub['recall_tuned_mean'].idxmax()
    terbaik = sub.loc[idx_max]
    r20 = sub[sub['rasio_uji'] == RASIO_SKRIPSI].iloc[0]
    print(f'{nama_model:<15} recall tertinggi pada split {terbaik["rasio_label"]} '
          f'({terbaik["recall_tuned_mean"]:.4f}); pada 80:20 = {r20["recall_tuned_mean"]:.4f} '
          f'(selisih {(terbaik["recall_tuned_mean"]-r20["recall_tuned_mean"])*100:.2f} poin, '
          f'std 80:20 = {r20["recall_tuned_std"]*100:.2f} poin)')
print('Selisih terhadap rasio terbaik yang lebih kecil daripada std antar seed')
print('berarti perbedaan tersebut TIDAK signifikan secara praktis.')

## Visualisasi 2 - Stabilitas estimasi vs ukuran data uji

Grafik ini adalah sisi lain dari trade-off. Sumbu-Y adalah **standar deviasi metrik
antar 5 seed** - yakni seberapa besar angka hasil uji berubah hanya karena data uji
kebetulan terbagi berbeda.

Semakin kecil data uji (kanan ke kiri), semakin **tidak stabil** estimasinya. Ini
alasan kenapa split 90:10 atau 95:5 berbahaya untuk skripsi: recall yang dilaporkan
bisa berubah beberapa poin hanya karena mengganti `random_state`.

In [ ]:
# ============================================================
# CELL 11: Visualisasi 2 - Stabilitas estimasi (std antar seed)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

for nama_model in DAFTAR_MODEL:
    sub = (df_split_ringkas[df_split_ringkas['model'] == nama_model]
           .sort_values('n_test'))
    axes[0].plot(sub['n_test'], sub['recall_tuned_std'] * 100,
                 marker='o', markersize=7, linewidth=2.2,
                 color=WARNA_MODEL[nama_model], label=nama_model)
    axes[1].plot(sub['rasio_uji'] * 100, sub['roc_auc_tuned_std'] * 100,
                 marker='s', markersize=7, linewidth=2.2,
                 color=WARNA_MODEL[nama_model], label=nama_model)

n_test_20 = int(df_split_ringkas[df_split_ringkas['rasio_uji'] == RASIO_SKRIPSI]['n_test'].mean())
axes[0].axvline(n_test_20, color=WARNA_AKSEN, linestyle='--', linewidth=2.2)
axes[0].text(n_test_20, 0.02, ' n_test 80:20', transform=axes[0].get_xaxis_transform(),
             color=WARNA_AKSEN, fontsize=10, fontweight='bold', va='bottom')
axes[0].set_xlabel('Jumlah sampel data uji (n_test)')
axes[0].set_ylabel('Std recall antar 5 seed (poin %)')
axes[0].set_title('Semakin kecil data uji, semakin liar estimasi recall', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].axvline(RASIO_SKRIPSI * 100, color=WARNA_AKSEN, linestyle='--', linewidth=2.2)
axes[1].set_xlabel('Proporsi data uji (%)')
axes[1].set_ylabel('Std ROC-AUC antar 5 seed (poin %)')
axes[1].set_title('Stabilitas ROC-AUC terhadap ukuran data uji', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Eksperimen 1 (lanjutan): Stabilitas Estimasi vs Ukuran Data Uji',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('split_stabilitas_estimasi')
plt.show()

garis('RINGKAS STABILITAS (std recall antar seed, poin %)')
piv = (df_split_ringkas.pivot(index='rasio_label', columns='model',
                              values='recall_tuned_std') * 100).round(3)
piv = piv.reindex([label_rasio(r) for r in DAFTAR_RASIO])
display(piv)
rata_std = df_split_ringkas.groupby('rasio_uji')['recall_tuned_std'].mean() * 100
print(f'Rata-rata std (3 model) pada 80:20 : {rata_std.loc[RASIO_SKRIPSI]:.3f} poin')
print(f'Rata-rata std (3 model) pada 90:10 : {rata_std.loc[0.10]:.3f} poin')
print(f'Rata-rata std (3 model) pada 50:50 : {rata_std.loc[0.50]:.3f} poin')

---

# EKSPERIMEN 2 - Analisis Margin of Error (argumen kuantitatif inti)

Ini adalah bagian terpenting untuk menjawab penguji, karena mengubah pertanyaan
"kenapa 80:20" dari selera menjadi **hitungan statistik**.

Recall pada dasarnya adalah sebuah **proporsi**: dari sekian pasien yang benar-benar
diabetes di data uji, berapa persen yang berhasil ditangkap model. Karena itu presisi
estimasinya mengikuti rumus standard error proporsi:

$$SE(\text{recall}) = \sqrt{\frac{\hat{p}(1-\hat{p})}{n_{\text{positif uji}}}},\qquad
MoE_{95\%} = 1.96 \times SE$$

Perhatikan penyebutnya: **bukan** jumlah seluruh sampel uji, melainkan **jumlah sampel
positif (diabetes) di data uji**. Pada dataset ini kelasnya timpang (~8.5% positif),
sehingga memperkecil data uji langsung memangkas jumlah kasus positif dan melebarkan
selang kepercayaan dengan cepat.

Ukuran sampel dihitung pada **dataset penuh (96.146 baris)** karena keputusan rasio ini
berlaku untuk pelatihan model final; nilai recall diambil dari hasil Eksperimen 1.

In [ ]:
# ============================================================
# CELL 12: Eksperimen 2 - Margin of Error 95% untuk recall
# ============================================================
MODEL_MOE     = 'Random Forest'          # model kandidat produksi
N_TOTAL_PENUH = int(len(X_all))
N_POS_PENUH   = int(y_all.sum())

garis('EKSPERIMEN 2: MARGIN OF ERROR ESTIMASI RECALL')
print(f'Model acuan          : {MODEL_MOE}')
print(f'Dataset penuh        : {N_TOTAL_PENUH:,} baris, {N_POS_PENUH:,} kasus positif '
      f'({N_POS_PENUH/N_TOTAL_PENUH*100:.2f}%)')
print('Rumus MoE 95%        : 1.96 * sqrt(p*(1-p)/n_positif_uji)')
print('-' * 70)

baris_moe = []
for rasio in DAFTAR_RASIO:
    n_test     = int(round(N_TOTAL_PENUH * rasio))
    n_train    = N_TOTAL_PENUH - n_test
    n_pos_test = int(round(N_POS_PENUH * rasio))
    r_row = df_split_ringkas[(df_split_ringkas['model'] == MODEL_MOE) &
                             (df_split_ringkas['rasio_uji'] == rasio)].iloc[0]
    recall = float(r_row['recall_tuned_mean'])
    ci_bawah, ci_atas, moe = ci95_proporsi(recall, n_pos_test)
    baris_moe.append({
        'rasio_uji'      : rasio,
        'rasio_label'    : label_rasio(rasio),
        'proporsi_latih' : round(1 - rasio, 2),
        'n_train'        : n_train,
        'n_test'         : n_test,
        'n_positif_test' : n_pos_test,
        'recall'         : recall,
        'ci_bawah'       : float(ci_bawah),
        'ci_atas'        : float(ci_atas),
        'moe_persen'     : float(moe * 100),
        'lebar_ci_persen': float((ci_atas - ci_bawah) * 100),
        'std_empiris_persen': float(r_row['recall_tuned_std'] * 100),
    })

df_moe = pd.DataFrame(baris_moe).sort_values('rasio_uji', ascending=False).reset_index(drop=True)
simpan_tabel(df_moe.round(4), 'tabel_margin_of_error', tampilkan=False)

print(f'{"Split":<8}{"n_train":>10}{"n_test":>9}{"n_pos_uji":>11}'
      f'{"recall":>9}{"CI bawah":>10}{"CI atas":>10}{"MoE(+/-%)":>11}')
print('-' * 78)
for _, r in df_moe.iterrows():
    tanda = '  <== dipakai skripsi' if abs(r['rasio_uji'] - RASIO_SKRIPSI) < 1e-9 else ''
    print(f'{r["rasio_label"]:<8}{int(r["n_train"]):>10,}{int(r["n_test"]):>9,}'
          f'{int(r["n_positif_test"]):>11,}{r["recall"]:>9.4f}'
          f'{r["ci_bawah"]:>10.4f}{r["ci_atas"]:>10.4f}{r["moe_persen"]:>11.2f}{tanda}')
print('-' * 78)

moe_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['moe_persen'].iloc[0])
moe_10 = float(df_moe[df_moe['rasio_uji'] == 0.10]['moe_persen'].iloc[0])
moe_50 = float(df_moe[df_moe['rasio_uji'] == 0.50]['moe_persen'].iloc[0])
npos_20 = int(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['n_positif_test'].iloc[0])
npos_10 = int(df_moe[df_moe['rasio_uji'] == 0.10]['n_positif_test'].iloc[0])
recall_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['recall'].iloc[0])
print(f'MoE 80:20 = +/-{moe_20:.2f}% dengan {npos_20:,} kasus positif di data uji')
print(f'MoE 90:10 = +/-{moe_10:.2f}% dengan {npos_10:,} kasus positif '
      f'(MoE membengkak {moe_10/moe_20:.2f}x)')
print(f'MoE 50:50 = +/-{moe_50:.2f}% (hanya {(1 - moe_50/moe_20)*100:.1f}% lebih sempit '
      f'daripada 80:20, tetapi mengorbankan '
      f'{int(df_moe[df_moe["rasio_uji"]==RASIO_SKRIPSI]["n_train"].iloc[0]) - int(df_moe[df_moe["rasio_uji"]==0.50]["n_train"].iloc[0]):,} baris data latih)')

In [ ]:
# ============================================================
# CELL 13: Grafik trade-off dua panel - Recall (atas) vs MoE (bawah)
#
# CATATAN VISUALISASI: gambar ini SENGAJA TIDAK memakai dua sumbu-y pada satu
# panel (twinx). Pada grafik dua sumbu, posisi relatif kedua kurva - termasuk
# titik potongnya - sepenuhnya ditentukan oleh pilihan rentang masing-masing
# sumbu, sehingga "persilangan" bisa dimunculkan di mana saja hanya dengan
# mengubah batas sumbu. Pembaca mudah menafsirkannya sebagai temuan, padahal
# itu artefak skala. Dua panel bertumpuk dengan sumbu-x IDENTIK menyampaikan
# trade-off yang sama dan dibandingkan secara vertikal tanpa distorsi skala.
# ============================================================
WARNA_MOE = '#8e44ad'
df_plot = df_moe.sort_values('proporsi_latih')
x             = (df_plot['proporsi_latih'] * 100).to_numpy(dtype=float)
recall_persen = (df_plot['recall'] * 100).to_numpy(dtype=float)
std_persen    = df_plot['std_empiris_persen'].to_numpy(dtype=float)
moe_persen    = df_plot['moe_persen'].to_numpy(dtype=float)
label_x       = list(df_plot['rasio_label'])

fig, (ax_a, ax_b) = plt.subplots(2, 1, sharex=True, figsize=(11, 8))

# --- PANEL ATAS: kualitas model (recall) ---------------------------------
ax_a.errorbar(x, recall_persen, yerr=std_persen, marker='s', markersize=8,
              linewidth=2.6, capsize=5, color=WARNA_MODEL[MODEL_MOE])
ax_a.set_ylabel(f'Recall {MODEL_MOE} (%)')
ax_a.set_title('Panel A - Kualitas model: recall terhadap porsi data latih\n'
               '(titik = rata-rata 5 seed, error bar = std antar seed)',
               fontsize=11.5, fontweight='bold', loc='left')
batas_bawah = float(np.min(recall_persen - std_persen))
batas_atas  = float(np.max(recall_persen + std_persen))
rentang_r   = max(batas_atas - batas_bawah, 0.5)
# Ruang kosong bawah untuk anotasi, ruang atas untuk label zona
ax_a.set_ylim(batas_bawah - rentang_r * 0.55, batas_atas + rentang_r * 0.45)

# --- PANEL BAWAH: presisi estimasi (margin of error) ---------------------
ax_b.bar(x, moe_persen, width=3.0, color=WARNA_MOE, alpha=0.85,
         edgecolor='white', linewidth=1.2)
for xi, val in zip(x, moe_persen):
    ax_b.text(xi, val, f'{val:.2f}', ha='center', va='bottom', fontsize=9)
ax_b.set_ylabel('Margin of Error 95% recall\n(+/- poin persen)')
ax_b.set_xlabel('Rasio split (data latih : data uji)')
ax_b.set_title('Panel B - Presisi estimasi: margin of error terhadap ukuran data uji\n'
               '(makin kecil data uji, makin lebar selang kepercayaan)',
               fontsize=11.5, fontweight='bold', loc='left')
ax_b.set_ylim(0, float(moe_persen.max()) * 1.35)
ax_b.set_xticks(x)
ax_b.set_xticklabels(label_x)

# --- Penanda rasio terpilih: IDENTIK di kedua panel ----------------------
for ax in (ax_a, ax_b):
    ax.axvspan(48.5, 65.0, color='#c0392b', alpha=0.05)
    ax.axvspan(87.5, 91.5, color='#c0392b', alpha=0.05)
    ax.axvspan(72.5, 85.0, color='#27ae60', alpha=0.06)
    ax.axvline(80, color=WARNA_AKSEN, linestyle='--', linewidth=2.6, alpha=0.9)

tr_a, tr_b = ax_a.get_xaxis_transform(), ax_b.get_xaxis_transform()
ax_a.text(56.5, 0.94, 'data latih terbuang\n-> model kurang optimal', transform=tr_a,
          ha='center', va='top', fontsize=9.5, color='#c0392b')
ax_b.text(89.5, 0.94, 'data uji terlalu kecil\n-> estimasi tidak presisi', transform=tr_b,
          ha='center', va='top', fontsize=9.5, color='#c0392b')

# Anotasi rasio terpilih: bentuk sama di kedua panel, hanya angkanya yang berbeda
ax_a.annotate(f'rasio terpilih 80:20\nrecall {recall_20*100:.2f}%',
              xy=(80, recall_20 * 100), xycoords='data',
              xytext=(0.05, 0.07), textcoords='axes fraction',
              fontsize=10.5, fontweight='bold', color=WARNA_AKSEN,
              arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, linewidth=2))
ax_b.annotate(f'rasio terpilih 80:20\nMoE +/-{moe_20:.2f} poin  |  '
              f'{npos_20:,} positif di data uji',
              xy=(80, moe_20), xycoords='data',
              xytext=(0.05, 0.80), textcoords='axes fraction',
              fontsize=10.5, fontweight='bold', color=WARNA_AKSEN,
              arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, linewidth=2))

plt.suptitle('Eksperimen 2: Trade-off Kualitas Model (Recall) vs Presisi Estimasi (MoE)',
             fontsize=14, fontweight='bold')
fig.align_ylabels([ax_a, ax_b])
plt.tight_layout()
simpan_gambar('split_margin_of_error')
plt.show()

garis('INTERPRETASI')
print('Kedua panel memakai sumbu-x yang sama, jadi tiap rasio dibaca lurus ke bawah.')
print('Panel A (recall) naik ke kanan lalu mendatar: data latih besar -> model makin baik,')
print('tetapi manfaatnya berhenti bertambah setelah sekitar 75-80% data latih.')
print('Panel B (MoE) naik ke kanan: data uji makin kecil -> estimasi makin tidak presisi.')
print('Pada 80:20, Panel A sudah berada di area datar sementara Panel B masih rendah,')
print('sehingga 80:20 menjadi kompromi rasional antara kedua kriteria tersebut.')
print('Catatan metodologis: kedua besaran sengaja dipisah ke dua panel, bukan ditumpuk')
print('pada satu grafik dua sumbu-y, agar tidak ada titik potong semu akibat skala.')

---

# EKSPERIMEN 3 - Learning Curve

Sweep rasio menunjukkan performa akhir per rasio; *learning curve* menunjukkan
**bentuk proses belajarnya**: bagaimana skor validasi berubah ketika jumlah data latih
dinaikkan bertahap dari 10% sampai 100%.

Konfigurasi: `sklearn.model_selection.learning_curve`, `scoring='recall'`,
`cv=StratifiedKFold(5, shuffle=True)`, `train_sizes = 10%, 20%, ..., 100%`.
Pita berbayang = ±1 standar deviasi antar 5 fold.

Yang dicari: **titik plateau**, yaitu ukuran data latih terkecil yang skor validasinya
sudah mencapai >= 99% dari skor maksimum. Bila plateau tercapai jauh sebelum 100% data
latih, maka menyisihkan 20% data untuk pengujian **tidak merugikan model**.

In [ ]:
# ============================================================
# CELL 14: Eksperimen 3 - Learning curve (recall) untuk 3 model
# ============================================================
TRAIN_SIZES = np.linspace(0.1, 1.0, 10)
cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

garis('EKSPERIMEN 3: LEARNING CURVE (scoring=recall, 5-fold stratified)')
print(f'Titik train_size : {[f"{t:.0%}" for t in TRAIN_SIZES]}')
print(f'Estimasi fit     : {len(DAFTAR_MODEL) * len(TRAIN_SIZES) * 5} kali pelatihan')
print('-' * 70)

hasil_lc   = {}
baris_lc   = []
plateau_lc = {}

for nama_model in DAFTAR_MODEL:
    t0 = time.time()
    model = PABRIK_MODEL[nama_model](pakai_smote=True)
    n_abs, skor_train, skor_val = learning_curve(
        model, X_eks, y_eks,
        train_sizes=TRAIN_SIZES, cv=cv_lc, scoring='recall',
        n_jobs=-1, shuffle=True, random_state=RANDOM_STATE)

    train_mean, train_std = skor_train.mean(axis=1), skor_train.std(axis=1)
    val_mean,   val_std   = skor_val.mean(axis=1),   skor_val.std(axis=1)

    val_maks    = float(val_mean.max())
    idx_plateau = int(np.argmax(val_mean >= 0.99 * val_maks))
    plateau_lc[nama_model] = {
        'fraksi_plateau'  : float(TRAIN_SIZES[idx_plateau]),
        'n_train_plateau' : int(n_abs[idx_plateau]),
        'recall_plateau'  : float(val_mean[idx_plateau]),
        'recall_maks'     : val_maks,
        'recall_100persen': float(val_mean[-1]),
    }
    hasil_lc[nama_model] = {
        'train_sizes_fraksi' : TRAIN_SIZES.tolist(),
        'train_sizes_absolut': n_abs.tolist(),
        'train_mean': train_mean.tolist(), 'train_std': train_std.tolist(),
        'val_mean'  : val_mean.tolist(),   'val_std'  : val_std.tolist(),
        **plateau_lc[nama_model],
    }
    for i, frac in enumerate(TRAIN_SIZES):
        baris_lc.append({
            'model': nama_model,
            'train_size_fraksi': round(float(frac), 2),
            'n_train': int(n_abs[i]),
            'recall_train_mean': float(train_mean[i]),
            'recall_train_std' : float(train_std[i]),
            'recall_val_mean'  : float(val_mean[i]),
            'recall_val_std'   : float(val_std[i]),
        })

    print(f'{nama_model:<15} selesai {time.time()-t0:>6.1f}s | '
          f'plateau pada {TRAIN_SIZES[idx_plateau]:.0%} data latih '
          f'({int(n_abs[idx_plateau]):,} baris) | '
          f'recall val 100% data = {val_mean[-1]:.4f}')

df_lc = pd.DataFrame(baris_lc)
simpan_tabel(df_lc.round(4), 'tabel_learning_curve', tampilkan=False)
display(df_lc.head(10))

In [ ]:
# ============================================================
# CELL 15: Plot learning curve (train vs validasi, pita +/- std)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5.6), sharey=False)

for ax, nama_model in zip(axes, DAFTAR_MODEL):
    h = hasil_lc[nama_model]
    n_abs      = np.array(h['train_sizes_absolut'])
    train_mean = np.array(h['train_mean']); train_std = np.array(h['train_std'])
    val_mean   = np.array(h['val_mean']);   val_std   = np.array(h['val_std'])
    warna = WARNA_MODEL[nama_model]

    ax.plot(n_abs, train_mean, marker='o', markersize=6, linewidth=2.2,
            color=warna, label='Recall data latih')
    ax.fill_between(n_abs, train_mean - train_std, train_mean + train_std,
                    color=warna, alpha=0.18)
    ax.plot(n_abs, val_mean, marker='s', markersize=6, linewidth=2.2,
            linestyle='--', color='#34495e', label='Recall validasi (5-fold)')
    ax.fill_between(n_abs, val_mean - val_std, val_mean + val_std,
                    color='#34495e', alpha=0.15)

    ax.axvline(h['n_train_plateau'], color=WARNA_AKSEN, linestyle=':', linewidth=2.4)
    ax.annotate(f'plateau: {h["fraksi_plateau"]:.0%} data latih',
                xy=(h['n_train_plateau'], h['recall_plateau']), xycoords='data',
                xytext=(0.40, 0.12), textcoords='axes fraction',
                fontsize=9, fontweight='bold', color=WARNA_AKSEN,
                arrowprops=dict(arrowstyle='->', color=WARNA_AKSEN, linewidth=1.6))

    # Catatan: dengan cv=5-fold, titik train_size 100% pada learning curve = 80%
    # dari seluruh data, jadi titik paling kanan setara persis dengan split 80:20.
    ax.text(0.985, 0.02, 'titik terkanan = 80% data\n(setara split 80:20)',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8.5, color='#7f8c8d')

    ax.set_title(nama_model, fontweight='bold', color=warna)
    ax.set_xlabel('Jumlah sampel data latih')
    ax.set_ylabel('Recall')
    ax.legend(fontsize=8.5, loc='lower left', framealpha=0.9)

plt.suptitle('Eksperimen 3: Learning Curve - Recall vs Jumlah Data Latih '
             '(pita = +/- 1 std antar fold)', fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('split_learning_curve')
plt.show()

garis('TITIK PLATEAU (train_size terkecil dengan recall validasi >= 99% recall maksimum)')
for nama_model in DAFTAR_MODEL:
    p = plateau_lc[nama_model]
    delta = (p['recall_100persen'] - p['recall_plateau']) * 100
    print(f'{nama_model:<15} plateau {p["fraksi_plateau"]:>4.0%} data latih '
          f'({p["n_train_plateau"]:>6,} baris) | recall plateau {p["recall_plateau"]:.4f} -> '
          f'recall 100% data {p["recall_100persen"]:.4f} (tambahan hanya {delta:+.2f} poin)')
print('-' * 70)
print('CATATAN PENTING: karena learning curve memakai 5-fold CV, titik train_size 100%')
print('pada kurva di atas setara dengan 80% dari seluruh data - yaitu persis porsi data')
print('latih pada skema 80:20. Jadi kurva ini langsung memotret skema yang dipakai skripsi.')
print('Kesimpulan: kurva validasi sudah datar jauh sebelum 100% data latih dipakai.')
print('Menyisihkan 20% data untuk pengujian TIDAK menurunkan kemampuan belajar model')
print('secara berarti, karena model sudah berada di area plateau.')

---

# EKSPERIMEN 4 - Apakah holdout 80:20 bias dibanding Cross-Validation?

Keberatan lanjutan yang wajar dari penguji: *"kalau begitu kenapa tidak pakai
cross-validation saja, bukan single holdout?"*

Untuk menjawabnya, estimasi recall/F1/AUC dari **holdout 80:20 (5 seed)** dibandingkan
dengan tiga skema cross-validation pada model Random Forest:

1. **5-Fold Stratified CV**
2. **10-Fold Stratified CV**
3. **RepeatedStratifiedKFold(5 fold x 5 repetisi)** = 25 evaluasi

Catatan metodologis: agar perbandingan **apple-to-apple**, metrik holdout di sini
memakai *threshold* default 0.5 - sama dengan yang dipakai `cross_validate` - bukan
threshold Youden.

Bila selisih rata-rata holdout terhadap Repeated CV kecil (dalam rentang 1 std),
maka estimasi holdout 80:20 **tidak bias** dan sah dipakai sebagai skema pelaporan
utama, dengan biaya komputasi jauh lebih murah.

In [ ]:
# ============================================================
# CELL 16: Eksperimen 4 - Holdout 80:20 vs 5-Fold vs 10-Fold vs Repeated CV
# ============================================================
SCORING_CV = {'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
MODEL_CV   = 'Random Forest'

garis('EKSPERIMEN 4: PERBANDINGAN SKEMA EVALUASI (model Random Forest)')
print('Metrik memakai threshold default 0.5 agar sebanding dengan cross_validate.')
print('-' * 70)

baris_skema = []

# --- Skema A: holdout 80:20 dengan 5 seed (hasil dari Eksperimen 1) --------
sub_hold = df_split_detail[(df_split_detail['model'] == MODEL_CV) &
                           (df_split_detail['rasio_uji'] == RASIO_SKRIPSI)]
baris_skema.append({
    'skema'        : 'Holdout 80:20 (5 seed)',
    'n_evaluasi'   : int(len(sub_hold)),
    'recall_mean'  : float(sub_hold['recall_default'].mean()),
    'recall_std'   : float(sub_hold['recall_default'].std(ddof=1)),
    'f1_mean'      : float(sub_hold['f1_default'].mean()),
    'f1_std'       : float(sub_hold['f1_default'].std(ddof=1)),
    'roc_auc_mean' : float(sub_hold['roc_auc_default'].mean()),
    'roc_auc_std'  : float(sub_hold['roc_auc_default'].std(ddof=1)),
    'waktu_s'      : float(sub_hold['waktu_latih_s'].sum()),
})
print(f'[1/4] Holdout 80:20 (5 seed)  -> recall {baris_skema[-1]["recall_mean"]:.4f} '
      f'+/-{baris_skema[-1]["recall_std"]:.4f} '
      f'({baris_skema[-1]["waktu_s"]:.1f}s, dipakai ulang dari Eksperimen 1)')

# --- Skema B, C, D: cross-validation --------------------------------------
SKEMA_CV = [
    ('5-Fold CV',            StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)),
    ('10-Fold CV',           StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)),
    ('Repeated 5-Fold x5 CV', RepeatedStratifiedKFold(n_splits=5, n_repeats=5,
                                                      random_state=RANDOM_STATE)),
]

for i, (nama_skema, cv_obj) in enumerate(SKEMA_CV, start=2):
    t0 = time.time()
    hasil_cv = cross_validate(buat_pipeline_rf(), X_eks, y_eks, cv=cv_obj,
                              scoring=SCORING_CV, n_jobs=-1)
    durasi = time.time() - t0
    baris_skema.append({
        'skema'        : nama_skema,
        'n_evaluasi'   : int(len(hasil_cv['test_recall'])),
        'recall_mean'  : float(hasil_cv['test_recall'].mean()),
        'recall_std'   : float(hasil_cv['test_recall'].std(ddof=1)),
        'f1_mean'      : float(hasil_cv['test_f1'].mean()),
        'f1_std'       : float(hasil_cv['test_f1'].std(ddof=1)),
        'roc_auc_mean' : float(hasil_cv['test_roc_auc'].mean()),
        'roc_auc_std'  : float(hasil_cv['test_roc_auc'].std(ddof=1)),
        'waktu_s'      : float(durasi),
    })
    print(f'[{i}/4] {nama_skema:<22} -> recall {baris_skema[-1]["recall_mean"]:.4f} '
          f'+/-{baris_skema[-1]["recall_std"]:.4f} ({durasi:.1f}s)')

df_holdout_cv = pd.DataFrame(baris_skema)
simpan_tabel(df_holdout_cv.round(4), 'tabel_holdout_vs_cv', tampilkan=False)

print('-' * 70)
print(f'{"Skema":<24}{"n":>4}{"recall":>18}{"F1":>18}{"ROC-AUC":>18}{"waktu(s)":>10}')
print('-' * 92)
for _, r in df_holdout_cv.iterrows():
    print(f'{r["skema"]:<24}{int(r["n_evaluasi"]):>4}'
          f'{r["recall_mean"]:>11.4f} +/-{r["recall_std"]:.4f}'
          f'{r["f1_mean"]:>11.4f} +/-{r["f1_std"]:.4f}'
          f'{r["roc_auc_mean"]:>11.4f} +/-{r["roc_auc_std"]:.4f}'
          f'{r["waktu_s"]:>10.1f}')
print('-' * 92)

ref = df_holdout_cv[df_holdout_cv['skema'] == 'Repeated 5-Fold x5 CV'].iloc[0]
hol = df_holdout_cv[df_holdout_cv['skema'] == 'Holdout 80:20 (5 seed)'].iloc[0]
selisih_recall = abs(hol['recall_mean'] - ref['recall_mean'])
selisih_auc    = abs(hol['roc_auc_mean'] - ref['roc_auc_mean'])
print(f'Selisih recall  holdout 80:20 vs Repeated CV : {selisih_recall*100:.2f} poin '
      f'(std Repeated CV = {ref["recall_std"]*100:.2f} poin)')
print(f'Selisih ROC-AUC holdout 80:20 vs Repeated CV : {selisih_auc*100:.2f} poin')
print(f'Rasio waktu komputasi Repeated CV : Holdout = '
      f'{ref["waktu_s"]/max(hol["waktu_s"], 1e-9):.1f}x lebih lama')
konsisten_cv = bool(selisih_recall <= ref['recall_std'])
if konsisten_cv:
    print('PUTUSAN: selisih berada di dalam 1 standar deviasi Repeated CV ->')
    print('         estimasi holdout 80:20 KONSISTEN dengan cross-validation (tidak bias).')
else:
    print('PUTUSAN: selisih melebihi 1 standar deviasi Repeated CV ->')
    print('         laporkan hasil holdout 80:20 BERSAMA hasil Repeated CV di skripsi.')

In [ ]:
# ============================================================
# CELL 17: Visualisasi Eksperimen 4 - Holdout vs Cross-Validation
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2.1, 1]})

METRIK_BAR = [('recall', 'Recall'), ('f1', 'F1-Score'), ('roc_auc', 'ROC-AUC')]
WARNA_BAR  = [WARNA_MODEL['Random Forest'], WARNA_AKSEN, '#8e44ad']
posisi = np.arange(len(df_holdout_cv))
lebar  = 0.26

for j, ((kunci, judul), warna) in enumerate(zip(METRIK_BAR, WARNA_BAR)):
    axes[0].bar(posisi + (j - 1) * lebar, df_holdout_cv[f'{kunci}_mean'], lebar,
                yerr=df_holdout_cv[f'{kunci}_std'], capsize=4,
                color=warna, alpha=0.88, label=judul,
                edgecolor='white', linewidth=1.2)
    for xi, val, sd in zip(posisi + (j - 1) * lebar,
                           df_holdout_cv[f'{kunci}_mean'],
                           df_holdout_cv[f'{kunci}_std']):
        axes[0].text(xi, val + sd + 0.022, f'{val:.3f}', ha='center', fontsize=8.5)

axes[0].axhline(hol['recall_mean'], color='#c0392b', linestyle=':', linewidth=1.8)
axes[0].text(-0.45, hol['recall_mean'] + 0.012, 'recall holdout 80:20',
             color='#c0392b', fontsize=9, ha='left', va='bottom')
axes[0].set_xticks(posisi)
axes[0].set_xticklabels(df_holdout_cv['skema'], rotation=12, fontsize=9.5)
axes[0].set_ylabel('Skor (threshold 0.5)')
axes[0].set_ylim(0, 1.18)
axes[0].set_title('Estimasi metrik tiap skema (error bar = std antar evaluasi)',
                  fontweight='bold')
axes[0].legend(fontsize=9, loc='upper right', ncol=3)

axes[1].bar(posisi, df_holdout_cv['waktu_s'],
            color=['#2ecc71' if s.startswith('Holdout') else '#95a5a6'
                   for s in df_holdout_cv['skema']],
            edgecolor='white', linewidth=1.2)
for xi, val in zip(posisi, df_holdout_cv['waktu_s']):
    axes[1].text(xi, val, f'{val:.1f}s', ha='center', va='bottom', fontsize=9)
axes[1].set_ylim(0, float(df_holdout_cv['waktu_s'].max()) * 1.18)
axes[1].set_xticks(posisi)
axes[1].set_xticklabels(df_holdout_cv['skema'], rotation=25, ha='right', fontsize=8.5)
axes[1].set_ylabel('Waktu komputasi (detik)')
axes[1].set_title('Biaya komputasi tiap skema', fontweight='bold')

plt.suptitle('Eksperimen 4: Validasi Skema Holdout 80:20 terhadap Cross-Validation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('split_holdout_vs_cv')
plt.show()

garis('INTERPRETASI')
print('Tinggi batang keempat skema praktis sama -> holdout 80:20 memberi estimasi')
print('yang setara dengan cross-validation, dengan waktu komputasi jauh lebih murah.')
print('Cross-validation tetap dipakai di notebook 04 sebagai validasi statistik lanjutan,')
print('sedangkan holdout 80:20 dipakai sebagai skema pelaporan utama.')

---

# KESIMPULAN - Skor Komposit dan Argumen Final

Rasio terbaik dipilih dengan **skor komposit** yang menggabungkan dua kriteria yang
tadi bertentangan, setelah keduanya dinormalisasi min-max ke rentang [0, 1]:

$$\text{skor}(r) = \text{recall}_{\text{norm}}(r) - \lambda \cdot \text{MoE}_{\text{norm}}(r),
\qquad \lambda = 1.0$$

Artinya: rasio yang baik adalah rasio yang memberi recall tinggi **dan sekaligus**
margin of error rendah. Rasio yang mengorbankan salah satunya secara ekstrem
(50:50 atau 90:10) otomatis mendapat skor rendah.

In [ ]:
# ============================================================
# CELL 18: Skor komposit, argumen final, dan penyimpanan JSON kontrak
# ============================================================
BOBOT_PENALTI_MOE = 1.0

def normalisasi(v):
    v = np.asarray(v, dtype=float)
    rentang = v.max() - v.min()
    return np.zeros_like(v) if rentang < 1e-12 else (v - v.min()) / rentang

df_skor = df_moe.sort_values('rasio_uji', ascending=False).reset_index(drop=True).copy()
df_skor['recall_norm'] = normalisasi(df_skor['recall'].values)
df_skor['moe_norm']    = normalisasi(df_skor['moe_persen'].values)
df_skor['skor_komposit'] = df_skor['recall_norm'] - BOBOT_PENALTI_MOE * df_skor['moe_norm']

idx_terbaik  = int(df_skor['skor_komposit'].idxmax())
rasio_komposit = float(df_skor.loc[idx_terbaik, 'rasio_uji'])
skor_terbaik   = float(df_skor.loc[idx_terbaik, 'skor_komposit'])
skor_20        = float(df_skor[df_skor['rasio_uji'] == RASIO_SKRIPSI]['skor_komposit'].iloc[0])
selisih_skor   = skor_terbaik - skor_20

simpan_tabel(df_skor[['rasio_label', 'n_train', 'n_test', 'n_positif_test', 'recall',
                      'moe_persen', 'recall_norm', 'moe_norm', 'skor_komposit']].round(4),
             'tabel_skor_komposit_rasio', tampilkan=False)

garis('SKOR KOMPOSIT PER RASIO (recall_norm - 1.0 x MoE_norm)')
for _, r in df_skor.iterrows():
    tanda = ' <== SKOR TERTINGGI' if abs(r['rasio_uji'] - rasio_komposit) < 1e-9 else ''
    tanda += ' [dipakai skripsi]' if abs(r['rasio_uji'] - RASIO_SKRIPSI) < 1e-9 else ''
    print(f'  {r["rasio_label"]:<7} recall={r["recall"]:.4f} '
          f'MoE=+/-{r["moe_persen"]:.2f}%  skor={r["skor_komposit"]:+.4f}{tanda}')

if abs(rasio_komposit - RASIO_SKRIPSI) < 1e-9:
    catatan_pilihan = ('Skor komposit tertinggi jatuh TEPAT pada rasio 80:20, sehingga '
                       'rasio yang dipakai skripsi terkonfirmasi secara kuantitatif.')
elif selisih_skor <= 0.10:
    catatan_pilihan = (f'Skor komposit tertinggi jatuh pada rasio '
                       f'{label_rasio(rasio_komposit)} dengan selisih hanya '
                       f'{selisih_skor:.4f} terhadap 80:20 (praktis setara, di bawah '
                       f'ambang 0.10). Rasio 80:20 tetap dipilih karena berada pada '
                       f'plateau performa, MoE-nya kecil, dan merupakan konvensi baku '
                       f'yang memudahkan pembandingan dengan penelitian lain.')
else:
    catatan_pilihan = (f'Skor komposit tertinggi jatuh pada rasio '
                       f'{label_rasio(rasio_komposit)} (selisih {selisih_skor:.4f} '
                       f'terhadap 80:20). Selisih ini perlu dilaporkan apa adanya di '
                       f'skripsi dan dibahas sebagai keterbatasan pemilihan rasio.')

RASIO_TERPILIH = RASIO_SKRIPSI

plateau_maks = max(p['fraksi_plateau'] for p in plateau_lc.values())
r20_rf = df_split_ringkas[(df_split_ringkas['model'] == MODEL_MOE) &
                          (df_split_ringkas['rasio_uji'] == RASIO_SKRIPSI)].iloc[0]
n_train_20 = int(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['n_train'].iloc[0])
n_test_20p = int(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['n_test'].iloc[0])
n_train_50 = int(df_moe[df_moe['rasio_uji'] == 0.50]['n_train'].iloc[0])
recall_50  = float(df_moe[df_moe['rasio_uji'] == 0.50]['recall'].iloc[0])
recall_20  = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['recall'].iloc[0])
recall_10  = float(df_moe[df_moe['rasio_uji'] == 0.10]['recall'].iloc[0])

delta_20_10 = (recall_10 - recall_20) * 100
std_20      = float(r20_rf['recall_tuned_std']) * 100
signifikan  = abs(delta_20_10) > std_20
frasa_sig   = ('lebih besar daripada' if signifikan else 'lebih kecil daripada')
kesan_sig   = ('sehingga perlu dilaporkan dan dibahas sebagai keterbatasan'
               if signifikan else 'sehingga tidak signifikan secara praktis')
ci_b_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['ci_bawah'].iloc[0])
ci_a_20 = float(df_moe[df_moe['rasio_uji'] == RASIO_SKRIPSI]['ci_atas'].iloc[0])
lebar_ci_20 = (ci_a_20 - ci_b_20) * 100
frasa_ci = ('cukup sempit untuk mendukung klaim ilmiah' if moe_20 <= 2.0
            else 'perlu dilaporkan apa adanya beserta lebarnya')
frasa_cv = ('sehingga estimasi holdout ini tidak bias'
            if konsisten_cv else
            'sehingga hasil holdout sebaiknya dilaporkan bersama hasil Repeated CV')

print()
garis('ARGUMEN FINAL: KENAPA RASIO SPLIT 80:20')
print('(1) PLATEAU PERFORMA')
print(f'    Learning curve menunjukkan recall validasi sudah mencapai >=99% nilai')
print(f'    maksimumnya pada {plateau_maks:.0%} data latih (paling lambat di antara 3 model).')
print(f'    Sweep rasio menegaskan hal yang sama: recall {MODEL_MOE} berubah dari '
      f'{recall_50:.4f} (50:50)')
print(f'    menjadi {recall_20:.4f} (80:20), lalu bergeser {delta_20_10:+.2f} poin di 90:10.')
print(f'    Pergeseran itu {frasa_sig} standar deviasi antar seed ({std_20:.2f} poin),')
print(f'    {kesan_sig}.')
print()
print('(2) PRESISI ESTIMASI MASIH TERJAGA PADA 20% DATA UJI')
print(f'    Data uji 20% = {n_test_20p:,} sampel, di antaranya {npos_20:,} kasus positif.')
print(f'    Margin of error 95% untuk recall = +/-{moe_20:.2f} poin persen.')
print(f'    Selang kepercayaan 95% recall = [{ci_b_20:.4f}, {ci_a_20:.4f}] '
      f'(lebar {lebar_ci_20:.2f} poin),')
print(f'    {frasa_ci}.')
print()
print('(3) RASIO EKSTREM MERUGIKAN DI KEDUA ARAH')
print(f'    - Data uji 10% : MoE membengkak menjadi +/-{moe_10:.2f} poin '
      f'({moe_10/moe_20:.2f}x lebih lebar)')
print(f'      karena hanya tersisa {npos_10:,} kasus positif untuk diuji, dan std antar seed naik.')
print(f'    - Data uji 40-50% : membuang {n_train_20 - n_train_50:,} baris data latih '
      f'dibanding 80:20')
print(f'      tanpa manfaat sepadan (MoE hanya menyempit {moe_20 - moe_50:.2f} poin).')
print()
print('(4) KONSISTEN DENGAN CROSS-VALIDATION DAN PRAKTIK BAKU')
print(f'    Estimasi holdout 80:20 berselisih {selisih_recall*100:.2f} poin recall dari')
print(f'    Repeated Stratified 5-Fold x5 CV (25 evaluasi, std {ref["recall_std"]*100:.2f} poin),')
print(f'    {frasa_cv}, dengan biaya komputasi '
      f'{ref["waktu_s"]/max(hol["waktu_s"], 1e-9):.1f}x lebih murah.')
print('    Rasio 80:20 juga selaras dengan prinsip Pareto (80/20) dan merupakan')
print('    konvensi yang lazim dipakai pada literatur machine learning kesehatan,')
print('    sehingga hasil penelitian ini dapat dibandingkan langsung dengan studi lain.')
print()
print('CATATAN PEMILIHAN:')
print(f'    {catatan_pilihan}')
print('=' * 70)

teks_kesimpulan = (
    f'Rasio split 80:20 dipilih berdasarkan empat bukti eksperimental. '
    f'(1) Learning curve menunjukkan recall validasi mencapai plateau (>=99% nilai maksimum) '
    f'pada {plateau_maks:.0%} data latih, sehingga menambah porsi latih melebihi 80% tidak lagi '
    f'meningkatkan kemampuan model secara berarti. '
    f'(2) Sweep tujuh rasio x lima seed x tiga model memperlihatkan recall {MODEL_MOE} '
    f'{recall_50:.4f} pada 50:50, {recall_20:.4f} pada 80:20, dan {recall_10:.4f} pada 90:10; '
    f'selisih 80:20 terhadap 90:10 ({delta_20_10:+.2f} poin) {frasa_sig} '
    f'standar deviasi antar seed ({std_20:.2f} poin), {kesan_sig}. '
    f'(3) Analisis margin of error menunjukkan data uji 20% ({n_test_20p:,} sampel, {npos_20:,} kasus '
    f'positif) memberi MoE 95% recall sebesar +/-{moe_20:.2f} poin persen, sedangkan data uji 10% '
    f'melebarkannya menjadi +/-{moe_10:.2f} poin ({moe_10/moe_20:.2f}x) dan data uji 50% hanya '
    f'menyempitkannya {moe_20 - moe_50:.2f} poin dengan mengorbankan {n_train_20 - n_train_50:,} baris '
    f'data latih. '
    f'(4) Estimasi holdout 80:20 berselisih {selisih_recall*100:.2f} poin recall dari '
    f'Repeated Stratified 5-Fold x5 Cross-Validation (std {ref["recall_std"]*100:.2f} poin), '
    f'{frasa_cv}, namun {ref["waktu_s"]/max(hol["waktu_s"], 1e-9):.1f}x lebih murah secara '
    f'komputasi. Dengan demikian 80:20 adalah titik kompromi optimal '
    f'antara kualitas model dan presisi estimasi kinerjanya, bukan sekadar mengikuti kebiasaan.'
)

teks_alasan = (
    f'Rasio {label_rasio(RASIO_TERPILIH)} stratified menyisakan {npos_20:,} kasus diabetes '
    f'di data uji ({n_test_20p:,} sampel). Dengan recall {recall_20:.3f}, margin of error 95% '
    f'(Wald) hanya +/-{moe_20:.2f} poin persen - cukup sempit untuk klaim skripsi - sementara '
    f'data latih tetap {n_train_20:,} baris, sudah berada di area plateau learning curve '
    f'({plateau_maks:.0%} data latih).'
)

hasil_split_ratio = {
    'rasio_terpilih'  : float(RASIO_TERPILIH),
    'rasio_label'     : label_rasio(RASIO_TERPILIH),
    'rasio_skor_komposit_terbaik': float(rasio_komposit),
    # --- ringkasan skalar (dipakai notebook 06 & website) ---
    'n_latih'         : int(n_train_20),
    'n_uji'           : int(n_test_20p),
    'n_positif_uji'   : int(npos_20),
    'margin_of_error_recall_pp': float(moe_20),
    'recall_acuan'    : float(recall_20),
    'model_acuan'     : MODEL_MOE,
    'alasan'          : teks_alasan,
    'skor_komposit'   : df_skor[['rasio_label', 'rasio_uji', 'recall', 'moe_persen',
                                 'recall_norm', 'moe_norm', 'skor_komposit']].to_dict('records'),
    'tabel'           : df_split_ringkas.to_dict('records'),
    'tabel_detail_ringkas': df_split_detail.groupby(['model', 'rasio_label'])['recall_tuned']
                              .agg(['mean', 'std', 'min', 'max']).reset_index().to_dict('records'),
    'learning_curve'  : hasil_lc,
    'plateau'         : plateau_lc,
    'margin_of_error' : df_moe.to_dict('records'),
    'holdout_vs_cv'   : df_holdout_cv.to_dict('records'),
    'kesimpulan'      : teks_kesimpulan,
    'metadata'        : {
        'mode_cepat'      : bool(MODE_CEPAT),
        'n_data_eksperimen': int(len(X_eks)),
        'n_data_penuh'    : int(N_TOTAL_PENUH),
        'n_positif_penuh' : int(N_POS_PENUH),
        'daftar_rasio'    : DAFTAR_RASIO,
        'daftar_seed'     : DAFTAR_SEED,
        'model_acuan'     : MODEL_MOE,
        'metrik_utama'    : 'recall (threshold Youden)',
        'bobot_penalti_moe': BOBOT_PENALTI_MOE,
        'catatan_pilihan' : catatan_pilihan,
    },
}
simpan_json(hasil_split_ratio, 'hasil_split_ratio')

garis('OUTPUT NOTEBOOK 01')
print('Tabel  : tabel_rasio_split_detail, tabel_rasio_split_ringkas,')
print('         tabel_margin_of_error, tabel_learning_curve, tabel_holdout_vs_cv,')
print('         tabel_skor_komposit_rasio')
print('Gambar : split_metrik_vs_rasio, split_stabilitas_estimasi,')
print('         split_margin_of_error, split_learning_curve, split_holdout_vs_cv')
print('JSON   : hasil_split_ratio  (dibaca oleh notebook 06)')
if MODE_CEPAT:
    print()
    print('PERINGATAN: notebook dijalankan dengan MODE_CEPAT = True.')
    print('Angka di atas adalah pratinjau pada subsample. Untuk angka final skripsi,')
    print('set MODE_CEPAT = False pada CELL 7 lalu jalankan ulang seluruh notebook.')

---

# RINGKASAN UNTUK SKRIPSI

Paragraf berikut siap disalin ke **Bab 3 (Metodologi, sub-bab Pembagian Data)** dan
dirujuk kembali di **Bab 4 (Hasil dan Pembahasan)**.

> **Wajib dibaca dulu:** angka yang tercetak miring pada paragraf Bab 4 di bawah
> (jumlah kasus positif, besaran *margin of error*, dan rasio pelebarannya) adalah
> nilai perkiraan. **Ganti seluruhnya dengan angka yang dicetak CELL 18** setelah
> notebook dijalankan dengan `MODE_CEPAT = False`.

---

### Untuk Bab 3 - Justifikasi Pembagian Data

> Pembagian data pada penelitian ini menggunakan rasio 80:20 (80% data latih, 20% data
> uji) dengan skema *stratified split* sehingga proporsi kelas diabetes terjaga pada
> kedua bagian. Rasio tersebut tidak ditetapkan berdasarkan kebiasaan, melainkan melalui
> pengujian empiris terhadap tujuh alternatif rasio (50:50, 60:40, 70:30, 75:25, 80:20,
> 85:15, dan 90:10), yang masing-masing diulang pada lima *random seed* berbeda dan
> diterapkan pada ketiga model (Random Forest, KNN, dan SVM). Pengujian ini dilengkapi
> analisis *learning curve*, perhitungan *margin of error* selang kepercayaan 95% pada
> metrik recall, serta pembandingan dengan skema *k-fold cross-validation*.

### Untuk Bab 4 - Hasil Pengujian Rasio Split

> Hasil pengujian menunjukkan bahwa kurva pembelajaran ketiga model telah mencapai
> kondisi jenuh (*plateau*) pada sekitar 70-80% data latih; recall validasi pada titik
> tersebut telah mencapai lebih dari 99% nilai maksimumnya, sehingga penambahan porsi
> data latih di atas 80% tidak lagi memberikan peningkatan kinerja yang berarti.
> Sebaliknya, analisis *margin of error* menunjukkan bahwa memperkecil data uji
> berdampak langsung pada presisi estimasi kinerja: dengan proporsi uji 20%, data uji
> memuat sekitar *1.696* kasus positif sehingga *margin of error* 95% untuk recall berada
> pada kisaran *±1,5 poin persen*, sedangkan proporsi uji 10% melebarkan *margin of error*
> tersebut sekitar *1,4 kali lipat* karena jumlah kasus positif yang diuji berkurang
> setengahnya. Di sisi lain, proporsi uji 40-50% membuang puluhan ribu baris data latih
> tanpa memberikan penyempitan *margin of error* yang sepadan. Pengujian tambahan
> membuktikan bahwa estimasi kinerja skema *holdout* 80:20 konsisten dengan estimasi
> *Repeated Stratified 5-Fold Cross-Validation* (selisih recall di bawah satu standar
> deviasi antar-lipatan), sehingga skema *holdout* 80:20 dapat dinilai tidak bias namun
> jauh lebih hemat secara komputasi. Dengan demikian rasio 80:20 merupakan titik
> kompromi optimal antara kualitas model dan presisi estimasi kinerjanya, sekaligus
> selaras dengan prinsip Pareto (80/20) dan konvensi yang lazim digunakan pada
> literatur *machine learning* di bidang kesehatan sehingga hasil penelitian ini dapat
> dibandingkan secara langsung dengan penelitian sejenis.

---

### Daftar keluaran notebook ini

| Jenis | Nama file |
|---|---|
| Tabel | `tabel_rasio_split_detail`, `tabel_rasio_split_ringkas`, `tabel_margin_of_error`, `tabel_learning_curve`, `tabel_holdout_vs_cv`, `tabel_skor_komposit_rasio` |
| Gambar | `split_metrik_vs_rasio`, `split_stabilitas_estimasi`, `split_margin_of_error`, `split_learning_curve`, `split_holdout_vs_cv` |
| JSON | `hasil_split_ratio` (kontrak untuk notebook `06`) |

### Catatan penting sebelum menulis angka ke skripsi

1. **Angka final wajib diambil dari run `MODE_CEPAT = False`.** Nilai yang muncul saat
   `MODE_CEPAT = True` dihitung pada subsample 30.000 baris dan hanya dimaksudkan untuk
   memastikan seluruh alur notebook berjalan; besaran *margin of error* pada tabel
   Eksperimen 2 memang sudah dihitung pada ukuran dataset penuh, tetapi nilai recall
   yang menjadi masukannya berasal dari subsample.
2. Set `PAKAI_DRIVE = True` pada CELL 2 bila hasil ingin dibaca oleh notebook
   `06_Model_Final_dan_Export_Produksi.ipynb`.
3. Sertakan gambar `split_margin_of_error` dan `split_learning_curve` di Bab 4 - dua
   gambar itulah bukti visual paling langsung untuk menjawab pertanyaan penguji.